In [ ]:
##### PACE-VCF Training Notebook
## XGBoost Full Model + Monte Carlo Feature Selection
## UPDATED: Native NaN handling (no median fill)

# ---

# ## Cell 1: Imports and Configuration

# =============================================================================
# PACE-VCF Training: XGBoost + Monte Carlo Feature Selection
# UPDATED: Use XGBoost native NaN handling instead of median fill
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
import rasterio
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import joblib
from datetime import datetime
import logging
import warnings
import time

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# =============================================================================
# CONFIGURATION
# =============================================================================

MODIS_TRAINING_DIR = Path("/explore/nobackup/projects/ilab/projects/MODIS-VCF/processedTiles/MOD44C/training")
PACE_BASE = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/output")
OUTPUT_DIR = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PACE_YEAR = 2025
TILE_SIZE_PACE = 600
NO_DATA = -10001

# Feature flags
INCLUDE_PACE_METRICS = True      # Include PACE_Metrics.tif (hyperspectral VIs)
INCLUDE_PACE_ALTSORT = True      # Include PACE_AltSort_Metrics.tif
INCLUDE_UNSORTED = False         # Include UnsortedMonthly* features (calendar-based)

# Model parameters
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Balanced sampling parameters
TARGET_BARE_PCT = 0.25
TARGET_LOW_PCT = 0.25
TARGET_FOREST_PCT = 0.08

# XGBoost GPU parameters (full model)
# UPDATED: Added missing=np.nan for native NaN handling
XGB_PARAMS = {
    'tree_method': 'gpu_hist',
    'n_estimators': 500,
    'max_depth': 10,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'verbosity': 1,
    'missing': np.nan,  # IMPORTANT: Tell XGBoost to handle NaN natively
}

# Monte Carlo parameters
N_TRIALS = 100
FEATURES_PER_TRIAL = 50
MIN_FEATURE_USAGE = 10
TOP_N_FEATURES = 50

# XGBoost parameters (faster for MC trials)
# UPDATED: Added missing=np.nan
XGB_TRIAL_PARAMS = {
    'tree_method': 'gpu_hist',
    'n_estimators': 100,
    'max_depth': 8,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': RANDOM_STATE,
    'verbosity': 0,
    'missing': np.nan,  # Native NaN handling
}

# XGBoost parameters (final MC model)
# UPDATED: Added missing=np.nan
XGB_FINAL_PARAMS = {
    'tree_method': 'gpu_hist',
    'n_estimators': 500,
    'max_depth': 10,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': RANDOM_STATE,
    'verbosity': 1,
    'missing': np.nan,  # Native NaN handling
}

print("="*70)
print("PACE-VCF TRAINING NOTEBOOK")
print("="*70)
print(f"  PACE metrics: {INCLUDE_PACE_METRICS}")
print(f"  AltSort metrics: {INCLUDE_PACE_ALTSORT}")
print(f"  Monte Carlo trials: {N_TRIALS}")
print(f"  Top-N features: {TOP_N_FEATURES}")
print(f"  NaN handling: NATIVE (XGBoost learns missing value behavior)")
print("="*70)


# =============================================================================
# Cell 2: Shared Functions - Data Loading
# =============================================================================

def load_or_extract_training_data(force_extract=True):
    """Load existing training data or extract fresh, including PACE metrics."""
    
    # Check for existing training data
    suffix = "_with_pace_altsort" if (INCLUDE_PACE_METRICS and INCLUDE_PACE_ALTSORT) else "_with_pace" if INCLUDE_PACE_METRICS else ""
    existing_files = list(OUTPUT_DIR.glob(f"training_data{suffix}_*.parquet"))
    
    if existing_files and not force_extract:
        latest = sorted(existing_files)[-1]
        logger.info(f"Loading existing training data: {latest}")
        return pd.read_parquet(latest)
    
    # Extract fresh
    logger.info("Extracting training data with PACE metrics...")
    
    parq_files = sorted(MODIS_TRAINING_DIR.glob("*.parq"))
    logger.info(f"  Found {len(parq_files)} training files")
    
    all_training = []
    modis_band_names = None
    pace_band_names = None
    altsort_band_names = None
    
    for f in parq_files:
        tile = f.stem.split('-')[0]
        
        modis_metrics_path = PACE_BASE / tile / str(PACE_YEAR) / "3-Metrics" / "MODIS_Metrics.tif"
        if not modis_metrics_path.exists():
            continue
        
        pace_metrics_path = PACE_BASE / tile / str(PACE_YEAR) / "3-Metrics" / "PACE_Metrics.tif"
        has_pace = INCLUDE_PACE_METRICS and pace_metrics_path.exists()
        
        altsort_metrics_path = PACE_BASE / tile / str(PACE_YEAR) / "3-Metrics" / "PACE_AltSort_Metrics.tif"
        has_altsort = INCLUDE_PACE_ALTSORT and altsort_metrics_path.exists()
        
        df = pd.read_parquet(f)
        df['pace_x'] = df['x'] // 8
        df['pace_y'] = df['y'] // 8
        
        agg = df.groupby(['pace_x', 'pace_y']).agg({'PercentTree': 'mean'}).reset_index()
        
        # Read MODIS metrics
        with rasterio.open(modis_metrics_path) as src:
            modis_bands = src.read()
            if modis_band_names is None:
                modis_band_names = [d if d else f"MODIS_Band_{i+1}" 
                                    for i, d in enumerate(src.descriptions)]
        
        # Read PACE metrics if available
        pace_bands = None
        if has_pace:
            with rasterio.open(pace_metrics_path) as src:
                pace_bands = src.read()
                if pace_band_names is None:
                    # Use band names directly - no prefix added
                    pace_band_names = [d if d else f"PACE_Band_{i+1}" 
                                       for i, d in enumerate(src.descriptions)]
        
        # Read AltSort metrics if available
        altsort_bands = None
        if has_altsort:
            with rasterio.open(altsort_metrics_path) as src:
                altsort_bands = src.read()
                if altsort_band_names is None:
                    # Use band names directly - no prefix added
                    altsort_band_names = [d if d else f"AltSort_Band_{i+1}" 
                                          for i, d in enumerate(src.descriptions)]
        
        for _, row in agg.iterrows():
            x, y = int(row['pace_x']), int(row['pace_y'])
            if 0 <= x < TILE_SIZE_PACE and 0 <= y < TILE_SIZE_PACE:
                modis_vals = modis_bands[:, y, x].astype(np.float32)
                modis_vals[modis_vals == NO_DATA] = np.nan
                
                sample = {'tile': tile, 'pace_x': x, 'pace_y': y, 
                          'PercentTree': row['PercentTree']}
                sample.update(dict(zip(modis_band_names, modis_vals)))
                
                if pace_bands is not None:
                    pace_vals = pace_bands[:, y, x].astype(np.float32)
                    pace_vals[pace_vals == NO_DATA] = np.nan
                    # NO PREFIX - use band names directly
                    sample.update(dict(zip(pace_band_names, pace_vals)))
                
                if altsort_bands is not None:
                    altsort_vals = altsort_bands[:, y, x].astype(np.float32)
                    altsort_vals[altsort_vals == NO_DATA] = np.nan
                    # NO PREFIX - use band names directly
                    sample.update(dict(zip(altsort_band_names, altsort_vals)))
                
                all_training.append(sample)
        
        status = []
        if has_pace: status.append("PACE")
        if has_altsort: status.append("AltSort")
        logger.info(f"    {tile}: {len(agg)} pixels {' ✓ ' + ', '.join(status) if status else ''}")
    
    result = pd.DataFrame(all_training)
    
    # Count feature types - UPDATED to match new naming
    # MODIS metrics don't start with PACEIndex or CrossIndex or Phenology etc.
    pace_prefixes = ('PACEIndex-', 'UnsortedMonthlyIndex-')
    altsort_prefixes = ('CrossIndex-', 'Phenology-', 'Brownest', 'mARI-sorted', 
                        'PRI-sorted', 'CCI-sorted', 'AtGreenUp', 'AtSenescence',
                        'AtPeakNDVI', 'AtMinNDVI', 'SeasonalRange')
    
    exclude_cols = ['tile', 'pace_x', 'pace_y', 'PercentTree']
    all_feature_cols = [c for c in result.columns if c not in exclude_cols]
    
    pace_cols = [c for c in all_feature_cols if c.startswith(pace_prefixes)]
    altsort_cols = [c for c in all_feature_cols if c.startswith(altsort_prefixes)]
    modis_cols = [c for c in all_feature_cols if c not in pace_cols and c not in altsort_cols]
    
    logger.info(f"  Total: {len(result):,} pixels")
    logger.info(f"  MODIS features: {len(modis_cols)}")
    logger.info(f"  PACE features: {len(pace_cols)}")
    logger.info(f"  AltSort features: {len(altsort_cols)}")
    
    # Filter samples with ≥50% features valid
    metric_cols = [c for c in result.columns if c not in exclude_cols]
    valid_counts = result[metric_cols].notna().sum(axis=1)
    result = result[valid_counts >= len(metric_cols) * 0.5]
    
    logger.info(f"  After filtering (≥50% features valid): {len(result):,} pixels")
    
    # Report NaN statistics
    nan_per_feature = result[metric_cols].isna().sum()
    features_with_nan = (nan_per_feature > 0).sum()
    max_nan_pct = (nan_per_feature / len(result) * 100).max()
    logger.info(f"  Features with any NaN: {features_with_nan}")
    logger.info(f"  Max NaN percentage in any feature: {max_nan_pct:.1f}%")
    
    # Save
    save_path = OUTPUT_DIR / f"training_data{suffix}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.parquet"
    result.to_parquet(save_path)
    logger.info(f"  Saved: {save_path}")
    
    return result


def apply_balanced_sampling(df, label_col='PercentTree', random_state=42):
    """
    Balanced sampling with configurable class weights.
    """
    np.random.seed(random_state)
    y = df[label_col].values
    
    bare_mask = y == 0
    low_mask = (y >= 1) & (y <= 25)
    med_mask = (y >= 26) & (y <= 50)
    high_mask = (y >= 51) & (y <= 80)
    forest_mask = (y >= 81) & (y <= 100)
    
    n_bare = np.sum(bare_mask)
    n_low = np.sum(low_mask)
    n_med = np.sum(med_mask)
    n_high = np.sum(high_mask)
    n_forest = np.sum(forest_mask)
    
    logger.info(f"Original: bare={n_bare:,}, low={n_low:,}, med={n_med:,}, high={n_high:,}, forest={n_forest:,}")
    
    target_med_high_pct = 1.0 - TARGET_BARE_PCT - TARGET_LOW_PCT - TARGET_FOREST_PCT
    fixed_med_high = n_med + n_high
    estimated_total = fixed_med_high / target_med_high_pct
    
    target_bare = min(int(estimated_total * TARGET_BARE_PCT), n_bare)
    target_low = min(int(estimated_total * TARGET_LOW_PCT), n_low)
    target_forest = int(estimated_total * TARGET_FOREST_PCT)
    
    keep_idx = []
    keep_idx += np.where(med_mask)[0].tolist()
    keep_idx += np.where(high_mask)[0].tolist()
    
    if target_bare <= n_bare:
        keep_idx += np.random.choice(np.where(bare_mask)[0], target_bare, replace=False).tolist()
    else:
        keep_idx += np.where(bare_mask)[0].tolist()
    
    if target_low <= n_low:
        keep_idx += np.random.choice(np.where(low_mask)[0], target_low, replace=False).tolist()
    else:
        keep_idx += np.where(low_mask)[0].tolist()
    
    forest_idx = np.where(forest_mask)[0]
    if target_forest > n_forest:
        keep_idx += np.random.choice(forest_idx, target_forest, replace=True).tolist()
        logger.info(f"  Forest UPSAMPLED: {n_forest:,} → {target_forest:,}")
    elif target_forest < n_forest:
        keep_idx += np.random.choice(forest_idx, target_forest, replace=False).tolist()
    else:
        keep_idx += forest_idx.tolist()
    
    df_balanced = df.iloc[keep_idx].reset_index(drop=True)
    
    y_bal = df_balanced[label_col].values
    n_total = len(y_bal)
    
    final_bare = np.sum(y_bal == 0)
    final_low = np.sum((y_bal >= 1) & (y_bal <= 25))
    final_med = np.sum((y_bal >= 26) & (y_bal <= 50))
    final_high = np.sum((y_bal >= 51) & (y_bal <= 80))
    final_forest = np.sum((y_bal >= 81) & (y_bal <= 100))
    
    logger.info(f"Balanced distribution:")
    logger.info(f"  Bare (0%):       {final_bare:>7,} ({100*final_bare/n_total:>5.1f}%) [target: {TARGET_BARE_PCT*100:.0f}%]")
    logger.info(f"  Low (1-25%):     {final_low:>7,} ({100*final_low/n_total:>5.1f}%) [target: {TARGET_LOW_PCT*100:.0f}%]")
    logger.info(f"  Medium (26-50%): {final_med:>7,} ({100*final_med/n_total:>5.1f}%)")
    logger.info(f"  High (51-80%):   {final_high:>7,} ({100*final_high/n_total:>5.1f}%)")
    logger.info(f"  Forest (81-100%):{final_forest:>7,} ({100*final_forest/n_total:>5.1f}%) [target: {TARGET_FOREST_PCT*100:.0f}%]")
    logger.info(f"Total: {len(df):,} → {n_total:,}")
    
    return df_balanced


def prepare_features(df, include_pace=True, include_altsort=True, include_unsorted=True):
    """
    Prepare features for training.
    
    UPDATED: 
    - No longer fills NaN with median - keeps NaN for XGBoost native handling.
    - Uses raw band names (no PACE_ or AltSort_ prefix added)
    """
    exclude = ['tile', 'pace_x', 'pace_y', 'PercentTree']
    feature_cols = [c for c in df.columns if c not in exclude]
    
    # Exclude QA metrics
    qa_features = [c for c in feature_cols if 'QA_' in c or c.startswith('QA')]
    feature_cols = [c for c in feature_cols if c not in qa_features]
    if qa_features:
        logger.info(f"  Excluded {len(qa_features)} QA metrics (diagnostic only)")
    
    # NOTE: AmpBandRefl-Band31 and ThermalGreenBrownDiff-Band31 used to be
    # excluded here as "problematic" (0% coverage) -- that was a bug in
    # MODIS_Metrics.tif generation (scale_thermal()'s absolute-temperature
    # gate wrongly applied to difference values), fixed in notebook 3l and
    # documented in skills/PACE-VCF.md. Tiles have been regenerated via 3l,
    # so these are real, valid features now -- no longer excluded.
    
    # Define prefixes for each source file
    pace_prefixes = ('PACEIndex-', 'UnsortedMonthlyIndex-')
    altsort_prefixes = ('CrossIndex-', 'Phenology-', 'Brownest', 'mARI-sorted', 
                        'PRI-sorted', 'CCI-sorted', 'AtGreenUp', 'AtSenescence',
                        'AtPeakNDVI', 'AtMinNDVI', 'SeasonalRange')
    
    # Optionally exclude UnsortedMonthly features
    if not include_unsorted:
        unsorted_before = len(feature_cols)
        feature_cols = [c for c in feature_cols if 'UnsortedMonthly' not in c]
        unsorted_removed = unsorted_before - len(feature_cols)
        if unsorted_removed > 0:
            logger.info(f"  Excluded {unsorted_removed} UnsortedMonthly features")
    
    # Optionally exclude PACE features
    if not include_pace:
        feature_cols = [c for c in feature_cols if not c.startswith(pace_prefixes)]
    
    # Optionally exclude AltSort features
    if not include_altsort:
        feature_cols = [c for c in feature_cols if not c.startswith(altsort_prefixes)]
    
    # Count by type (for logging)
    pace_features = [c for c in feature_cols if c.startswith(pace_prefixes)]
    altsort_features = [c for c in feature_cols if c.startswith(altsort_prefixes)]
    modis_features = [c for c in feature_cols if c not in pace_features and c not in altsort_features]
    unsorted_features = [c for c in feature_cols if 'UnsortedMonthly' in c]
    
    logger.info(f"  MODIS features: {len(modis_features)}")
    logger.info(f"  PACE features: {len(pace_features)}")
    logger.info(f"  AltSort features: {len(altsort_features)}")
    logger.info(f"  Unsorted monthly features: {len(unsorted_features)}")
    logger.info(f"  Total features: {len(feature_cols)}")
    
    # Keep NaN for XGBoost native handling
    X = df[feature_cols].copy()
    y = df['PercentTree'].values
    
    # Report NaN statistics
    nan_counts = X.isna().sum()
    features_with_nan = (nan_counts > 0).sum()
    total_nan = nan_counts.sum()
    total_values = X.shape[0] * X.shape[1]
    
    logger.info(f"  X shape: {X.shape}")
    logger.info(f"  y range: {y.min():.1f}% to {y.max():.1f}%, mean: {y.mean():.1f}%")
    logger.info(f"  NaN handling: NATIVE (XGBoost will learn missing value behavior)")
    logger.info(f"    Features with NaN: {features_with_nan}/{len(feature_cols)}")
    logger.info(f"    Total NaN values: {total_nan:,} ({100*total_nan/total_values:.2f}% of all values)")
    
    if features_with_nan > 0:
        top_nan = nan_counts[nan_counts > 0].sort_values(ascending=False).head(10)
        logger.info(f"    Top features by NaN count:")
        for feat, count in top_nan.items():
            logger.info(f"      {feat}: {count:,} ({100*count/len(X):.1f}%)")
    
    return X, y, feature_cols


print("Data loading functions defined ✓")


# =============================================================================
# Cell 3: XGBoost Full Model Functions
# =============================================================================

def train_xgboost(X, y, feature_names):
    """
    Train XGBoost with GPU using all features.
    
    UPDATED: Uses native NaN handling (no median fill).
    """
    
    logger.info("Training XGBoost (GPU) - Full Model...")
    logger.info(f"  Parameters: {XGB_PARAMS}")
    logger.info(f"  NaN handling: NATIVE (missing=np.nan)")
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    
    logger.info(f"  Train: {len(y_train):,}, Test: {len(y_test):,}")
    
    # Report NaN in train/test
    train_nan = X_train.isna().sum().sum()
    test_nan = X_test.isna().sum().sum()
    logger.info(f"  Train NaN values: {train_nan:,}")
    logger.info(f"  Test NaN values: {test_nan:,}")
    
    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)
    
    y_pred = model.predict(X_test)
    
    r_pearson, _ = pearsonr(y_test, y_pred)
    metrics = {
        'r2': r_pearson ** 2,  # Pearson correlation squared
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
        'mae': mean_absolute_error(y_test, y_pred),
        'best_iteration': model.best_iteration if hasattr(model, 'best_iteration') else XGB_PARAMS['n_estimators']
    }
    
    logger.info(f"\n  Results:")
    logger.info(f"    R²:   {metrics['r2']:.4f}")
    logger.info(f"    RMSE: {metrics['rmse']:.2f}%")
    logger.info(f"    MAE:  {metrics['mae']:.2f}%")
    
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    logger.info(f"\n  Top 20 features:")
    for _, row in importance_df.head(20).iterrows():
        logger.info(f"    {row['feature']:<45} {row['importance']:.4f}")
    
    return model, metrics, importance_df, (X_test, y_test, y_pred)


def visualize_xgb_results(metrics, importance_df, y_test, y_pred):
    """Create visualizations for XGBoost full model."""
    
    logger.info("Creating visualizations...")
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].scatter(y_test, y_pred, alpha=0.1, s=5)
    axes[0].plot([0, 100], [0, 100], 'r--', linewidth=2)
    axes[0].set_xlabel('Actual Tree Cover (%)')
    axes[0].set_ylabel('Predicted Tree Cover (%)')
    axes[0].set_title(f'Predicted vs Actual (R²={metrics["r2"]:.3f})')
    axes[0].set_xlim(0, 100)
    axes[0].set_ylim(0, 100)
    
    top20 = importance_df.head(20)
    axes[1].barh(range(len(top20)), top20['importance'].values)
    axes[1].set_yticks(range(len(top20)))
    axes[1].set_yticklabels(top20['feature'].values, fontsize=8)
    axes[1].invert_yaxis()
    axes[1].set_xlabel('Importance')
    axes[1].set_title('Top 20 Feature Importances')
    
    residuals = y_pred - y_test
    axes[2].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
    axes[2].axvline(0, color='r', linestyle='--')
    axes[2].set_xlabel('Residual (Predicted - Actual)')
    axes[2].set_ylabel('Count')
    axes[2].set_title(f'Residuals (RMSE={metrics["rmse"]:.2f}%)')
    
    plt.tight_layout()
    fig_path = OUTPUT_DIR / "xgb_full_model_results.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    logger.info(f"  Saved: {fig_path}")
    plt.show()


def save_xgb_results(model, metrics, importance_df, feature_names):
    """Save XGBoost full model results."""
    
    logger.info("Saving results...")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    model_path = OUTPUT_DIR / f"pace_vcf_xgb_full_{timestamp}.joblib"
    joblib.dump(model, model_path)
    
    xgb_path = OUTPUT_DIR / f"pace_vcf_xgb_full_{timestamp}.json"
    model.save_model(xgb_path)
    
    importance_df.to_csv(OUTPUT_DIR / f"xgb_full_importance_{timestamp}.csv", index=False)
    
    with open(OUTPUT_DIR / f"xgb_full_metrics_{timestamp}.txt", 'w') as f:
        f.write("PACE-VCF XGBoost Full Model Results\n")
        f.write("="*50 + "\n")
        f.write(f"Timestamp: {timestamp}\n")
        f.write(f"R²: {metrics['r2']:.4f}\n")
        f.write(f"RMSE: {metrics['rmse']:.2f}%\n")
        f.write(f"MAE: {metrics['mae']:.2f}%\n")
        f.write(f"Features: {len(feature_names)}\n")
        f.write(f"NaN handling: NATIVE (XGBoost learns missing value behavior)\n")
    
    logger.info(f"  Model: {model_path}")
    return model_path


print("XGBoost full model functions defined ✓")


# =============================================================================
# Cell 4: Monte Carlo Functions
# =============================================================================

def run_single_trial(trial_num, X_train, y_train, X_test, y_test, all_features):
    """Run a single Monte Carlo trial."""
    np.random.seed(RANDOM_STATE + trial_num)
    
    selected_features = np.random.choice(
        all_features, size=min(FEATURES_PER_TRIAL, len(all_features)), replace=False
    )
    
    model = xgb.XGBRegressor(**XGB_TRIAL_PARAMS)
    model.fit(X_train[selected_features], y_train, verbose=False)
    
    y_pred = model.predict(X_test[selected_features])
    
    r_pearson, _ = pearsonr(y_test, y_pred)
    r2 = r_pearson ** 2
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    importances = dict(zip(selected_features, model.feature_importances_))
    
    return {'trial': trial_num, 'r2': r2, 'rmse': rmse, 
            'features': list(selected_features), 'importances': importances}


def run_monte_carlo(X, y, feature_names):
    """
    Run Monte Carlo simulation to find best features.
    
    UPDATED: Uses native NaN handling.
    """
    logger.info("\nRunning Monte Carlo simulation...")
    logger.info(f"  NaN handling: NATIVE (XGBoost learns missing value behavior)")
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    
    logger.info(f"  Train: {len(y_train):,}, Test: {len(y_test):,}")
    logger.info(f"  Running {N_TRIALS} trials with {FEATURES_PER_TRIAL} features each...")
    
    all_trials = []
    feature_usage = {f: 0 for f in feature_names}
    feature_importance_sum = {f: 0.0 for f in feature_names}
    feature_importance_count = {f: 0 for f in feature_names}
    
    start_time = time.time()
    
    for trial_num in range(N_TRIALS):
        result = run_single_trial(trial_num, X_train, y_train, X_test, y_test, feature_names)
        all_trials.append(result)
        
        for feat in result['features']:
            feature_usage[feat] += 1
            if feat in result['importances']:
                feature_importance_sum[feat] += result['importances'][feat]
                feature_importance_count[feat] += 1
        
        if (trial_num + 1) % 10 == 0:
            elapsed = time.time() - start_time
            avg_r2 = np.mean([t['r2'] for t in all_trials])
            logger.info(f"    Trial {trial_num + 1}/{N_TRIALS}: avg R²={avg_r2:.4f}, elapsed={elapsed:.1f}s")
    
    logger.info(f"  Completed {N_TRIALS} trials in {time.time() - start_time:.1f}s")
    
    avg_importance = {f: feature_importance_sum[f] / feature_importance_count[f] 
                      if feature_importance_count[f] > 0 else 0.0 for f in feature_names}
    
    trials_df = pd.DataFrame([{'trial': t['trial'], 'r2': t['r2'], 'rmse': t['rmse']} for t in all_trials])
    importance_df = pd.DataFrame([{'feature': f, 'avg_importance': avg_importance[f], 
                                   'usage_count': feature_usage[f]} for f in feature_names]
                                 ).sort_values('avg_importance', ascending=False)
    
    return trials_df, importance_df, (X_train, X_test, y_train, y_test)


def select_top_features(importance_df, min_usage=MIN_FEATURE_USAGE, top_n=TOP_N_FEATURES):
    """Select top-N features that meet minimum usage threshold."""
    logger.info("\nSelecting top features...")
    
    qualified = importance_df[importance_df['usage_count'] >= min_usage].copy()
    logger.info(f"  Features meeting min usage ({min_usage}): {len(qualified)}")
    
    top_features = qualified.head(top_n)['feature'].tolist()
    
    logger.info(f"  Selected top {len(top_features)} features:")
    for _, row in qualified.head(top_n).iterrows():
        logger.info(f"    {row['feature']:<45} imp={row['avg_importance']:.4f}, used={row['usage_count']}")
    
    return top_features


def train_final_mc_model(X_train, y_train, X_test, y_test, top_features):
    """
    Train final model with selected features.
    
    UPDATED: Uses native NaN handling.
    """
    logger.info(f"\nTraining final model with {len(top_features)} features...")
    logger.info(f"  NaN handling: NATIVE")
    
    model = xgb.XGBRegressor(**XGB_FINAL_PARAMS)
    model.fit(X_train[top_features], y_train, eval_set=[(X_test[top_features], y_test)], verbose=50)
    
    y_pred = model.predict(X_test[top_features])
    
    r_pearson, _ = pearsonr(y_test, y_pred)
    metrics = {
        'r2': r_pearson ** 2,  # Pearson correlation squared -- consistent with train_xgboost()/run_single_trial()
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
        'mae': np.mean(np.abs(y_test - y_pred))
    }
    
    logger.info(f"\n  Final Model Results:")
    logger.info(f"    R²:   {metrics['r2']:.4f}")
    logger.info(f"    RMSE: {metrics['rmse']:.2f}%")
    logger.info(f"    MAE:  {metrics['mae']:.2f}%")
    
    final_importance = pd.DataFrame({
        'feature': top_features, 'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    logger.info(f"\n  Final Feature Importance:")
    for _, row in final_importance.iterrows():
        logger.info(f"    {row['feature']:<45} {row['importance']:.4f}")
    
    return model, metrics, final_importance, (y_test, y_pred)


def visualize_monte_carlo(trials_df, importance_df, final_importance, y_test, y_pred, metrics):
    """Create visualization of Monte Carlo results."""
    logger.info("\nCreating visualizations...")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    
    axes[0, 0].hist(trials_df['r2'], bins=30, edgecolor='black', alpha=0.7)
    axes[0, 0].axvline(trials_df['r2'].mean(), color='r', linestyle='--', label=f'Mean: {trials_df["r2"].mean():.3f}')
    axes[0, 0].axvline(metrics['r2'], color='g', linestyle='-', linewidth=2, label=f'Final: {metrics["r2"]:.3f}')
    axes[0, 0].set_xlabel('R² Score')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title(f'Monte Carlo Trial R² Distribution (n={len(trials_df)})')
    axes[0, 0].legend()
    
    top30_imp = importance_df.head(30)
    axes[0, 1].scatter(top30_imp['usage_count'], top30_imp['avg_importance'], alpha=0.7, s=50)
    for _, row in top30_imp.head(10).iterrows():
        axes[0, 1].annotate(row['feature'].split('-')[-1], (row['usage_count'], row['avg_importance']), fontsize=8)
    axes[0, 1].set_xlabel('Usage Count')
    axes[0, 1].set_ylabel('Average Importance')
    axes[0, 1].set_title('Feature Usage vs Importance (Top 30)')
    axes[0, 1].axvline(MIN_FEATURE_USAGE, color='r', linestyle='--', alpha=0.5)
    
    axes[1, 0].barh(range(len(final_importance)), final_importance['importance'].values)
    axes[1, 0].set_yticks(range(len(final_importance)))
    axes[1, 0].set_yticklabels(final_importance['feature'].values, fontsize=8)
    axes[1, 0].invert_yaxis()
    axes[1, 0].set_xlabel('Importance')
    axes[1, 0].set_title(f'Final Model Feature Importance ({len(final_importance)} features)')
    
    axes[1, 1].scatter(y_test, y_pred, alpha=0.1, s=5)
    axes[1, 1].plot([0, 100], [0, 100], 'r--', linewidth=2)
    axes[1, 1].set_xlabel('Actual Tree Cover (%)')
    axes[1, 1].set_ylabel('Predicted Tree Cover (%)')
    axes[1, 1].set_title(f'Final Model: Predicted vs Actual (R²={metrics["r2"]:.3f})')
    axes[1, 1].set_xlim(0, 100)
    axes[1, 1].set_ylim(0, 100)
    
    plt.tight_layout()
    fig_path = OUTPUT_DIR / "monte_carlo_results.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    logger.info(f"  Saved: {fig_path}")
    plt.show()


def save_mc_results(model, metrics, trials_df, importance_df, final_importance, top_features):
    """Save Monte Carlo results."""
    logger.info("\nSaving results...")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    model_path = OUTPUT_DIR / f"pace_vcf_mc_xgb_{timestamp}.json"
    model.save_model(model_path)
    
    trials_df.to_csv(OUTPUT_DIR / f"mc_trials_{timestamp}.csv", index=False)
    importance_df.to_csv(OUTPUT_DIR / f"mc_all_importance_{timestamp}.csv", index=False)
    final_importance.to_csv(OUTPUT_DIR / f"mc_final_importance_{timestamp}.csv", index=False)
    
    with open(OUTPUT_DIR / f"mc_top_features_{timestamp}.txt", 'w') as f:
        for feat in top_features:
            f.write(f"{feat}\n")
    
    with open(OUTPUT_DIR / f"mc_metrics_{timestamp}.txt", 'w') as f:
        f.write("PACE-VCF Monte Carlo Feature Selection Results\n")
        f.write("="*60 + "\n")
        f.write(f"Timestamp: {timestamp}\n")
        f.write(f"N_trials: {N_TRIALS}\n")
        f.write(f"Features_per_trial: {FEATURES_PER_TRIAL}\n")
        f.write(f"Top_N_features: {TOP_N_FEATURES}\n")
        f.write(f"NaN handling: NATIVE (XGBoost learns missing value behavior)\n")
        f.write(f"\nFinal Model:\n")
        f.write(f"  R²:   {metrics['r2']:.4f}\n")
        f.write(f"  RMSE: {metrics['rmse']:.2f}%\n")
        f.write(f"  MAE:  {metrics['mae']:.2f}%\n")
    
    logger.info(f"  Model: {model_path}")
    return model_path


print("Monte Carlo functions defined ✓")


# =============================================================================
# Cell 5: Load and Prepare Data (Run Once)
# =============================================================================

start_time = time.time()

# Load data (set force_extract=True to re-extract with new features)
df = load_or_extract_training_data(force_extract=True)

# Apply balanced sampling
df_balanced = apply_balanced_sampling(df, label_col='PercentTree')

# Prepare features - control what's included here
# UPDATED: No longer fills NaN - keeps them for XGBoost native handling
X, y, feature_cols = prepare_features(
    df_balanced, 
    include_pace=INCLUDE_PACE_METRICS, 
    include_altsort=INCLUDE_PACE_ALTSORT,
    include_unsorted=INCLUDE_UNSORTED
)

# Summary of feature types
print(f"\n{'='*60}")
print("FEATURE SUMMARY")
print(f"{'='*60}")

unsorted_features = [c for c in feature_cols if 'UnsortedMonthly' in c]
phenology_features = [c for c in feature_cols if any(x in c for x in 
    ['AtGreenUp', 'Senescence', 'AtPeakNDVI', 'AtMinNDVI', 'SeasonalRange', 
     'Brownest', 'Stressed', 'Chlorophyll', 'Carotenoid', 'Anthocyanin'])]
modis_style = [c for c in feature_cols if not c.startswith('PACE_') and not c.startswith('AltSort_')]

print(f"  MODIS-style features: {len(modis_style)}")
print(f"  PACE features: {len([c for c in feature_cols if c.startswith('PACE_')])}")
print(f"  AltSort features: {len([c for c in feature_cols if c.startswith('AltSort_')])}")
print(f"  Unsorted monthly: {len(unsorted_features)} {'(EXCLUDED)' if not INCLUDE_UNSORTED else '(included)'}")
print(f"  Phenological sorts: {len(phenology_features)}")
print(f"  TOTAL: {len(feature_cols)}")
print(f"{'='*60}")
print(f"  NaN handling: NATIVE (XGBoost will learn missing value behavior)")
print(f"{'='*60}")

print(f"\nData prepared in {time.time() - start_time:.1f}s")
print(f"  Samples: {len(y):,}")
print(f"  Features: {len(feature_cols)}")

In [ ]:
# # Check if thermal/snow features are in the final feature_cols
# thermal_in_features = [c for c in feature_cols if 'LST' in c or 'Band31' in c or 'Thermal' in c]
# snow_in_features = [c for c in feature_cols if 'Snow' in c]

# print(f"Thermal features in training: {len(thermal_in_features)}")
# for t in thermal_in_features:
#     print(f"  - {t}")

# print(f"\nSnow features in training: {len(snow_in_features)}")
# for s in snow_in_features:
#     print(f"  - {s}")

# print(f"\nTotal features for training: {len(feature_cols)}")

In [ ]:
#Cell 6: Run XGBoost Full Model# =============================================================================
# RUN XGBOOST FULL MODEL
# =============================================================================

print("="*70)
print("XGBOOST FULL MODEL")
print("="*70)

start_time = time.time()

# Train
model_full, metrics_full, importance_full, (X_test, y_test, y_pred_full) = train_xgboost(X, y, feature_cols)

# Visualize
visualize_xgb_results(metrics_full, importance_full, y_test, y_pred_full)

# Save
model_path_full = save_xgb_results(model_full, metrics_full, importance_full, feature_cols)

elapsed = time.time() - start_time

print("\n" + "="*70)
print("XGBOOST FULL MODEL COMPLETE!")
print("="*70)
print(f"  R²:   {metrics_full['r2']:.4f}")
print(f"  RMSE: {metrics_full['rmse']:.2f}%")
print(f"  MAE:  {metrics_full['mae']:.2f}%")
print(f"  Features: {len(feature_cols)}")
print(f"  Time: {elapsed/60:.1f} minutes")
print(f"  Model: {model_path_full}")
print("="*70)

In [ ]:
# Cell 7: Run Monte Carlo Feature Selection# =============================================================================
# RUN MONTE CARLO FEATURE SELECTION
# =============================================================================

print("="*70)
print("MONTE CARLO FEATURE SELECTION")
print("="*70)

start_time = time.time()

# Run Monte Carlo
trials_df, importance_df, (X_train_mc, X_test_mc, y_train_mc, y_test_mc) = run_monte_carlo(X, y, feature_cols)

# Select top features
top_features = select_top_features(importance_df)

# Train final model
model_mc, metrics_mc, final_importance, (y_test_mc, y_pred_mc) = train_final_mc_model(
    X_train_mc, y_train_mc, X_test_mc, y_test_mc, top_features
)

# Visualize
visualize_monte_carlo(trials_df, importance_df, final_importance, y_test_mc, y_pred_mc, metrics_mc)

# Save
model_path_mc = save_mc_results(model_mc, metrics_mc, trials_df, importance_df, final_importance, top_features)

total_time = time.time() - start_time

print("\n" + "="*70)
print("MONTE CARLO FEATURE SELECTION COMPLETE!")
print("="*70)
print(f"  Trials: {N_TRIALS}")
print(f"  Trial avg R²: {trials_df['r2'].mean():.4f} ± {trials_df['r2'].std():.4f}")
print(f"  Final model R²: {metrics_mc['r2']:.4f}")
print(f"  Final model RMSE: {metrics_mc['rmse']:.2f}%")
print(f"  Features selected: {len(top_features)}")
print(f"  Total time: {total_time/60:.1f} minutes")
print(f"  Model: {model_path_mc}")
print("="*70)


In [ ]:
# Cell 8: Compare Results# =============================================================================
# COMPARE RESULTS
# =============================================================================

print("="*70)
print("MODEL COMPARISON")
print("="*70)
print(f"\n{'Model':<25} {'R²':>10} {'RMSE':>10} {'MAE':>10} {'Features':>10}")
print("-"*70)
print(f"{'XGBoost Full':<25} {metrics_full['r2']:>10.4f} {metrics_full['rmse']:>10.2f} {metrics_full['mae']:>10.2f} {len(feature_cols):>10}")
print(f"{'Monte Carlo (Top 50)':<25} {metrics_mc['r2']:>10.4f} {metrics_mc['rmse']:>10.2f} {metrics_mc['mae']:>10.2f} {len(top_features):>10}")
print("-"*70)

r2_diff = metrics_full['r2'] - metrics_mc['r2']
feature_reduction = (1 - len(top_features)/len(feature_cols)) * 100

print(f"\nTrade-off:")
print(f"  R² reduction: {r2_diff:.4f} ({r2_diff/metrics_full['r2']*100:.1f}%)")
print(f"  Feature reduction: {feature_reduction:.1f}%")
print("="*70)
